[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bethavila/dncp-riesgo-contrataciones/blob/main/notebooks/00_pipeline_completo.ipynb)

# Detección de riesgo en contrataciones públicas (DNCP, 2020–2026) — pipeline completo
**Proyecto III Big Data — Maestría en Ciencia de Datos (UCOM)** · Bethania Avila, Arturo Fernández, Gustavo Bogado

Este notebook ejecuta el sistema de punta a punta y mide el tiempo de cada etapa (objetivo: decisiones en minutos).

| Etapa | Objetivo del TP | V de Big Data | Qué hace |
|---|---|---|---|
| captura | 1. Captura multifuente | Variedad, Velocidad | OCDS masivo 2020–2026, protestas DNCP, Banco Mundial (y la API DNCP en el notebook 05) |
| silver | 2. Almacenamiento escalable | Volumen, Veracidad | JSON anidado → 7 tablas Parquet particionadas; huellas SHA-256 e inventario de volumen |
| gold | 3. Procesamiento en minutos | Volumen | Indicadores de riesgo por etapa con SQL (DuckDB) |
| modelos | 4. Analítica predictiva | — | Validación de banderas, 3 detectores de anomalías comparados, riesgo de impugnación, SHAP |
| redes / series | 4. Analítica predictiva | — | Colusión entre proveedores (NetworkX) y pronóstico de tendencias |

In [ ]:
# Configuración del entorno (Google Colab o local)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/bethavila/dncp-riesgo-contrataciones.git"  # <- URL del repositorio en GitHub
USAR_DRIVE = False  # True: guarda el data lake en Google Drive y no repite descargas entre sesiones

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/dncp-riesgo-contrataciones")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements-colab.txt")], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["DNCP_DATA_DIR"] = "/content/drive/MyDrive/dncp_data_lake"
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dncp_riesgo import graficos as gr
from dncp_riesgo.pipeline import asegurar_datos, preparar

cfg, lake = preparar()
gr.estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| años:", cfg["fuentes"]["ocp_registry"]["anios"])

## Ejecución completa
Descarga ~0,8 GB comprimidos, normaliza ~100.000 procesos y ~10 millones de ítems, calcula los indicadores, entrena y compara los modelos y puntúa cada proceso.

In [ ]:
from dncp_riesgo.pipeline import ejecutar
resultados = ejecutar()
tiempos = resultados["tiempos"]
tiempos

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
t = tiempos.iloc[::-1]
ax.barh(t["etapa"].str.replace("Etapa ", ""), t["minutos"], color=gr.SERIES[0], height=0.6)
for i, v in enumerate(t["minutos"]):
    ax.text(v, i, f" {v:.1f} min".replace(".", ","), va="center", fontsize=8, color=gr.TINTA_2)
ax.set_xlabel("minutos"); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title(f"Tiempo por etapa — total {tiempos['minutos'].sum():.1f} minutos".replace(".", ","))
plt.show()

In [ ]:
lake.resumen()

## Resultados clave
### Comparación de detectores de anomalías (prueba temporal 2025)

In [ ]:
res = resultados["modelos"]
print("Detector elegido:", res.artefactos.nombre_detector, "—", res.decision_detector)
res.evaluacion_detectores

### Riesgo de impugnación (supervisado) frente a líneas base

In [ ]:
print("Modelo elegido:", res.artefactos.nombre_supervisado, "—", res.decision_supervisado)
res.evaluacion_supervisados

### Perfil del 5 % de mayor prioridad de auditoría (2025)

In [ ]:
res.perfil_prioridad

In [ ]:
cols = ["prioridad", "nivel", "comprador_nombre", "proveedor_nombre", "metodo_grupo", "monto_adjudicado", "banderas_activas", "factores_anomalia"]
res.puntajes.query("periodo == 'prueba'").sort_values("prioridad", ascending=False)[cols].head(15)